In [ ]:
SCHEMA_VERSION = "1.0"

NODE_SCHEMA = {
    "Recipe": {
        "meaning": "특정 출처에서 수집한 개별 레시피",
        "properties": {
            "recipe_uid": "STRING",
            "title": "STRING",
            "description": "STRING",
            "servings": "STRING",
            "cooking_time": "STRING",
            "difficulty": "STRING",
            "views": "INTEGER",
            "source_url": "STRING",
            "source": "STRING",
        },
        "required": [
            "recipe_uid",
            "title",
            "source",
            "source_url",
        ],
        "unique": ["recipe_uid"],
        "nullable": [
            "description",
            "servings",
            "cooking_time",
            "difficulty",
            "views",
        ],
        "description": {
            "recipe_uid": "출처와 원본 ID 등을 이용해 코드에서 생성한 고유 ID",
            "title": "원본 레시피 제목",
            "description": "원본 소개 문구. LLM이 생성한 요약이 아님",
            "servings": "원문 인분 표현. 예: 2~3인분",
            "cooking_time": "원문 조리 시간 표현. 예: 30분 이내",
            "difficulty": "원본에 표시된 난이도",
            "views": "수집 시점의 조회수. 미상은 null",
            "source_url": "원본 레시피 URL",
            "source": "코드에서 통일한 출처 이름",
        },
    },

    "Dish": {
        "meaning": "여러 개별 레시피를 묶는 표준 음식 개념",
        "properties": {
            "name": "STRING",
        },
        "required": ["name"],
        "unique": ["name"],
        "nullable": [],
        "description": {
            "name": "팀의 음식 분류 및 정규화 규칙에 따른 대표 음식명",
        },
    },

    "Ingredient": {
        "meaning": "여러 레시피에서 공유하는 정규화된 재료 개념",
        "properties": {
            "name": "STRING",
            "name_normalized": "STRING",
        },
        "required": [
            "name",
            "name_normalized",
        ],
        "unique": ["name_normalized"],
        "nullable": [],
        "description": {
            "name": "사용자에게 보여줄 대표 재료명",
            "name_normalized": (
                "정규화 및 ER을 거쳐 확정한 재료 식별용 이름. "
                "문서별 원문 표현은 Component.raw_name에 저장"
            ),
        },
    },

    "Component": {
        "meaning": "특정 레시피에 속하는 개별 재료 사용 항목",
        "properties": {
            "component_uid": "STRING",
            "raw_name": "STRING",
            "role": "STRING",
            "group": "STRING",
            "quantity": "STRING",
            "unit": "STRING",
            "preparation": "STRING",
            "detail": "STRING",
            "index": "INTEGER",
            "is_required": "BOOLEAN",
            "alternative_mode": "STRING",
            "evidence": "STRING",
            "quality_flags": "LIST<STRING>",
        },
        "required": [
            "component_uid",
            "raw_name",
            "role",
            "index",
            "alternative_mode",
            "evidence",
            "quality_flags",
        ],
        "unique": ["component_uid"],
        "nullable": [
            "group",
            "quantity",
            "unit",
            "preparation",
            "detail",
            "is_required",
        ],
        "enum": {
            "role": [
                "food",
                "seasoning",
                "unknown",
            ],
            "alternative_mode": [
                "none",
                "replacement",
                "either_or",
                "unresolved",
            ],
        },
        "defaults": {
            "role": "unknown",
            "alternative_mode": "none",
            "quality_flags": [],
        },
        "description": {
            "component_uid": (
                "코드에서 생성하는 항목 고유 ID. "
                "예: 10000recipe:6880798:component:001"
            ),
            "raw_name": "해당 항목의 원문 재료 표현. 예: 다진 마늘",
            "role": (
                "레시피 문맥에서의 재료/조미료 분류. "
                "food=재료, seasoning=조미료, unknown=판단 불가"
            ),
            "group": (
                "원문 재료 섹션 이름. "
                "예: 양념장, 고명, 밑간. 원문에 없으면 null"
            ),
            "quantity": (
                "원문 수량 표현. 예: 1/2, 2~3, 약간. "
                "숫자로 강제 변환하지 않음"
            ),
            "unit": (
                "원문 단위. 예: 모, 쪽, 큰술. "
                "불명확한 단위를 추정하지 않음"
            ),
            "preparation": "재료 손질·준비 상태. 예: 다진, 채 썬, 껍질 제거",
            "detail": (
                "다른 필드로 표현하지 못한 원문 조건이나 추가 설명. "
                "선택 여부와 대체재를 이 필드에만 저장하지 않음"
            ),
            "index": (
                "Recipe 내부에서 1부터 부여하는 항목 순번. "
                "조리 단계 번호가 아님"
            ),
            "is_required": (
                "true=필수라는 명시적 근거 있음, "
                "false=생략 가능이라는 명시적 근거 있음, "
                "null=판단 근거 없음"
            ),
            "alternative_mode": (
                "none=대체 표현 없음, "
                "replacement=기본 재료와 대체재가 구분됨, "
                "either_or=대등한 선택지, "
                "unresolved=현재 구조로 확정하기 어려움"
            ),
            "evidence": (
                "기본 재료 사용 항목을 뒷받침하는 원문 구절. "
                "수량·선택 여부 등 추출한 속성의 근거도 포함"
            ),
            "quality_flags": (
                "검토가 필요한 품질 경고 코드 목록. "
                "문제가 발견되지 않았으면 빈 목록"
            ),
        },
    },
}


RELATION_SCHEMA = {
    "VARIANT_OF": {
        "source": "Recipe",
        "target": "Dish",
        "meaning": "개별 레시피가 해당 표준 음식의 한 버전임",
        "properties": {},
        "validation_rules": [
            "현재 단일 음식 레시피 범위에서는 Recipe당 0~1개",
            "음식 분류가 불명확하면 임의로 연결하지 않고 검토 대상으로 남김",
        ],
    },

    "HAS_COMPONENT": {
        "source": "Recipe",
        "target": "Component",
        "meaning": "레시피에 속하는 개별 재료 사용 항목",
        "properties": {},
        "validation_rules": [
            "각 Component는 정확히 하나의 Recipe에 속함",
            "서로 다른 Recipe가 동일 Component를 공유하지 않음",
            "동일 Recipe 안에서 Component.index가 중복되지 않음",
        ],
    },

    "INGREDIENT": {
        "source": "Component",
        "target": "Ingredient",
        "meaning": "재료 사용 항목이 가리키는 기본 재료",
        "properties": {},
        "validation_rules": [
            "검증을 통과한 Component마다 정확히 1개",
            "대등한 선택지에서는 원문상 첫 재료를 기본 연결로 사용",
            "첫 재료라는 이유로 더 권장되는 재료라고 해석하지 않음",
        ],
    },

    "ALTERNATIVE": {
        "source": "Component",
        "target": "Ingredient",
        "meaning": (
            "해당 레시피의 해당 항목에서 기본 재료 대신 "
            "사용할 수 있다고 원문에 명시된 재료"
        ),
        "properties": {
            "raw_name": "STRING",
            "quantity": "STRING",
            "unit": "STRING",
            "preparation": "STRING",
            "condition": "STRING",
            "evidence": "STRING",
        },
        "required": [
            "raw_name",
            "evidence",
        ],
        "nullable": [
            "quantity",
            "unit",
            "preparation",
            "condition",
        ],
        "description": {
            "raw_name": "대체 재료의 원문 표현",
            "quantity": "원문에 별도로 명시된 대체재 수량",
            "unit": "원문에 별도로 명시된 대체재 단위",
            "preparation": "대체재에 별도로 명시된 손질·준비 상태",
            "condition": "대체가 가능한 조건. 원문에 없으면 null",
            "evidence": "기본 재료와 대체재의 대체 관계를 뒷받침하는 원문",
        },
        "validation_rules": [
            "원문에 대체 또는 선택 표현이 명시된 경우에만 생성",
            "대체재 수량이 없다고 기본 재료 수량을 자동 복사하지 않음",
            "ER 후 기본 Ingredient와 같아지는 연결은 검토",
            "여러 재료를 함께 사용해야 하는 복합 대체는 자동 확정하지 않음",
            "상호 대체·전역 대체·연쇄 대체 가능성을 자동 추론하지 않음",
        ],
    },
}


QUALITY_FLAG_CODES = [
    "AMBIGUOUS_INGREDIENT",
    "AMBIGUOUS_ROLE",
    "AMBIGUOUS_QUANTITY",
    "QUANTITY_CONFLICT",
    "AMBIGUOUS_ALTERNATIVE",
    "COMPOUND_ALTERNATIVE",
    "CONFLICTING_REQUIRED_STATUS",
    "EVIDENCE_NOT_FOUND",
    "POSSIBLE_DUPLICATE_COMPONENT",
]

10


In [68]:
CHECK_QUERY = """
MATCH (r:Recipe)
RETURN count(r) AS recipes
"""

with driver.session() as session:

    record = session.run(
        CHECK_QUERY
    ).single()

    print(record["recipes"])

10


In [69]:
query = """
MATCH (d:Dish)
WHERE toLower(trim(d.name)) IN [
    'unknown',
    'unknow',
    'none',
    'null',
    'n/a'
]
RETURN d.name AS name
"""

with driver.session() as session:
    result = list(session.run(query))

print(result)

[]


In [70]:
query = """
MATCH (r:Recipe)
WHERE NOT (r)-[:VARIANT_OF]->(:Dish)
RETURN count(r) AS recipes_without_dish
"""

with driver.session() as session:
    result = session.run(query).single()

print(
    "Dish 없는 Recipe:",
    result["recipes_without_dish"]
)

Dish 없는 Recipe: 6


In [71]:
BATCH_SIZE = 500

batch = []
total = 0

with driver.session() as session:

    with path.open("r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            raw_row = json.loads(line)

            graph_row = prepare_graph_row(raw_row)

            batch.append(graph_row)

            if len(batch) >= BATCH_SIZE:

                session.run(
                    LOAD_QUERY,
                    rows=batch
                ).consume()

                total += len(batch)

                print(f"{total:,}개 처리")

                batch = []

        # 마지막 남은 데이터
        if batch:

            session.run(
                LOAD_QUERY,
                rows=batch
            ).consume()

            total += len(batch)

print(f"전체 적재 완료: {total:,}개")

500개 처리
1,000개 처리
1,500개 처리
2,000개 처리
2,500개 처리
3,000개 처리
3,500개 처리
4,000개 처리
4,500개 처리
5,000개 처리
5,500개 처리
6,000개 처리
6,500개 처리
7,000개 처리
7,500개 처리
8,000개 처리
8,500개 처리
9,000개 처리
9,500개 처리
전체 적재 완료: 9,558개


In [ ]:
{
    "source": "10000recipe",
    "source_id": "7081897",
    "source_url": "https://www.10000recipe.com/recipe/7081897",
    "title": "전자레인지 다이어터를 위한 양배추 미소무침 만들기",
    "description": "다이어트 할때 채소를 많이 먹지만, 저는 그중에서도 양배추를 많이 먹어요. 그래서 매번 장을 볼때 빼놓지 않고 사두는데요. 보통은 양배추를 찜기에 쪄서 쌈으로 먹지만, 오늘은 좀 더 맛있게 무쳐서 밥이랑 먹고 싶어서 만든 양배추 미소무침 만들기입니다. 아직 더위가 채 가시지 않아서 찜기를 쓰기가 힘들어서 전자레인지로 간단하게 조리했고요. 재료도 간단하고 만들기도 쉬워서 누구나 쉽게 만들 수 있는 반찬이랍니다.",
    "servings": "2인분",
    "cooking_time": "10분 이내",
    "difficulty": "아무나",
    "ingredients": [
        {"group": "[재료]", "name": "양배추", "amount": "200g", "raw": "양배추 200g"}
    ],
    "seasonings": [
        {
            "group": "[미소양념]",
            "name": "다진마늘",
            "amount": "1/2숟갈",
            "raw": "다진마늘 1/2숟갈",
        },
        {
            "group": "[미소양념]",
            "name": "고춧가루",
            "amount": "1/2숟갈",
            "raw": "고춧가루 1/2숟갈",
        },
        {
            "group": "[미소양념]",
            "name": "진간장 미소 맛에 따라 선택",
            "amount": "미소 맛에 따라 선택",
            "raw": "진간장 미소 맛에 따라 선택 1/2숟갈",
        },
        {
            "group": "[미소양념]",
            "name": "미소된장 살짝 볼록하게",
            "amount": "살짝 볼록하게",
            "raw": "미소된장 살짝 볼록하게 1숟갈",
        },
        {
            "group": "[미소양념]",
            "name": "참기름",
            "amount": "2/3숟갈",
            "raw": "참기름 2/3숟갈",
        },
    ],
    "tools": [],
    "steps": [
        {
            "order": 1,
            "text": "먼저 양배추는 한입 크기로 깍둑썰어주세요. 그리고 깨끗하게 씻어서 물기를 가볍게 탁 털어줍니다. 이때 물기를 완벽하게 제거하면 안되고 꼭 양배추 전체적으로 물기가 남아 있어야해요.",
        },
        {
            "order": 2,
            "text": "씻어준 양배추는 전자레인지에 돌려도 되는 용기에 넣고 뚜껑을 덮어줍니다. 이때 반드시 숨구멍을 열어주셔야해요. 저처럼 용기가 있으면 사용하시고 없으면 전자레인지 돌려도 되는 그릇에 담아서 랩 씌우고 구멍을 몇개 뚫어주시면 됩니다. 그대로 약 2분 30초~3분간 전자레인지에 돌려주세요. 너무 오래 익으면 식감이 안좋고 양배추즙 마실때 특유의 이상한 냄새가 나거든요. 그래서 2분 30초 돌려보고 많이 딱딱하면 추가로 조금더 돌려주세요.",
        },
        {
            "order": 3,
            "text": "잘쪄진 양배추는 바로 체에 담아서 찬물에 열기를 완전히 식혀주세요. 그리고 손으로 물기를 꾹 짜줍니다. 이때 너무 과하게 짜주면 식감도 별로고 뻣뻣해지니까요. 적당히 꾹 짜주세요.",
        },
        {
            "order": 4,
            "text": "잘 짜준 양배추를 그릇에 담아주시고요. 여기에 다진마늘 1/2숟갈, 고춧가루 1/2숟갈 진간장 1/2숟갈을 넣어줍니다. 사용하시는 미소가 짜면 진간장을 줄여주시거나 아예 빼셔도 됩니다. 미소 된장마다 단맛이 강한게 있고 짠맛이 강한게 있거든요.",
        },
        {
            "order": 5,
            "text": "미소된장 1숟갈(살짝 볼록하게)도 넣고 골고루 버무려주세요. 그리고 간을 봅니다. 여기서 짜면 찐 양배추를 추가하시고요. 싱거우면 미소된장만 조금 더 추가합니다. 양배추는 찌면 단맛이 많이 올라와서 설탕이나 올리고당등은 따로 넣지 않아도 충분히 맛있어요. 다이어트 중이니까 당류를 제한하기 위함도 있습니다.",
        },
        {
            "order": 6,
            "text": "입맛에 맞게 됐으면 참기름 2/3숟갈 휙 두르고 마지막으로 숟가락으로 가볍게 골고루 섞어주시면 됩니다.",
        },
        {"order": 7, "text": "완성~!"},
    ],
    "created_at": "2026-08-23",
    "updated_at": "2026-08-24",
    "views": 56,
    "selection_reason": "latest",
}